In [ ]:
import os
import pandas as pd
import pickle
import re
import subprocess
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

In [ ]:
RESULTS_PATH = "results" # replace with your results directory

# Retrieving Data

In [ ]:
def count_total_csv_entries():
    import concurrent.futures
    from functools import partial
    
    def count_rows_in_file(file, prefix):
        if not file.endswith('.csv'):
            return 0
        file_path = prefix + "/" + file
        result = subprocess.run(['wc', '-l', file_path], capture_output=True, text=True)
        try:
            # Subtract 1 for header row
            return int(result.stdout.strip().split()[0]) - 1
        except (ValueError, IndexError):
            print(f"Error processing file: {file}")
            return 0
    
    total_rows = 0
    prefix = RESULTS_PATH
    class_files = os.listdir(prefix)
    
    process_file = partial(count_rows_in_file, prefix=prefix)
    with concurrent.futures.ThreadPoolExecutor(max_workers=6) as executor:
        future_to_file = {executor.submit(process_file, file): file for file in class_files}
        
        for i, future in enumerate(concurrent.futures.as_completed(future_to_file)):
            file = future_to_file[future]
            try:
                row_count = future.result()
                total_rows += row_count
                print(f'file: {file} {i+1}/{len(class_files)} - {row_count:,} rows')
            except Exception as e:
                print(f'Error processing {file}: {e}')
    
    print(f"Total number of entries across all CSV files: {total_rows:,}")
    return total_rows
count_total_csv_entries()

file: swh:1:snp:4abbe315eb9008f3c36fe2439ad5a4419fffa90c.csv 1/28 - 281,620,543 rows
file: swh:1:snp:77fdf23a7b93c3e477085e63a8228514a5141d81.csv 2/28 - 34,350,683 rows
file: checkpoints 3/28 - 0 rows
file: swh:1:snp:7bdd5a506a8c1bf163fd368523c8b24946cd0d08.csv 4/28 - 159,343,744 rows
file: swh:1:snp:90833b1d9246c51d17ad6d8c1cdc60d584444bd5.csv 5/28 - 182,357,327 rows
file: swh:1:snp:72dbef05d425ee828596e74a92fa6270d955359d.csv 6/28 - 180,747,424 rows
file: swh:1:snp:fa2b95d2e6fc91d0777af2a30c32c427fd559c24.csv 7/28 - 243,766,898 rows
file: focus 8/28 - 0 rows
file: swh:1:snp:f55d9473c9a69f86738c3478c858d55eb7edb65b.csv 9/28 - 161,276,043 rows
file: swh:1:snp:5a5f7a892f39485694421edc0f4aea88f98028a3.csv 10/28 - 163,024,881 rows
file: swh:1:snp:f62a0d62af412170e9ffd95e61a7c87b0ac3f292.csv 11/28 - 67,380,399 rows
file: swh:1:snp:183d6e31cf83bba4a9590045efa949d42b2ba0aa.csv 12/28 - 199,990,826 rows
file: swh:1:snp:2cbadedf325fa41a309aee0f6b9b1aa02a1c85e2.csv 13/28 - 100,183,412 rows
file:

12542848352

## Loading visit type

In [2]:
db = {}
with open('visit_type.pkl', 'rb') as f:
    db = pickle.load(f)

## Loading stars

In [30]:
db_stars = {}
with open('stars_without_dup.pkl', 'rb') as f:
    db_stars = pickle.load(f)

In [ ]:
print(f'Amount of repositories in the GitHub repo/star dataset: {len(db_stars):,}')

Amount of repositories in the GitHub repo/star dataset: 278,827,680


True

## Loading results

In [ ]:
prefix = RESULTS_PATH + "/focus/classes"
class_files = os.listdir(prefix)
df = pd.DataFrame()
for file in class_files:
    df = pd.concat([df, pd.read_csv(prefix + "/" + file, delimiter=";")])
    #df = pd.read_csv(file, delimiter=";")

## Filtering
We only keep `git` origin and we remove all branches named `HEAD` because it is an alias.

In [ ]:
print("size of all results unfiltered: ", len(df))

size of all results unfiltered:  10264390


True

In [ ]:
df = df[(df['branch_name'] != "HEAD")]
print("size of results when removing \'HEAD\' from branch: ", len(df))

size of results when removing 'HEAD' from branch:  9844211


True

In [ ]:
origins = df['origin'].drop_duplicates().reset_index(drop=True)
print(f'amount of origins with at least 1 root cause commits: {len(origins):,}')

amount of origins with at least 1 root cause commits: 1,237,353


True

In [ ]:
df = df[(df['origin'].map(db) == 'git')]
print(f'size of results when keeping only git: {len(df):,}')

size of results when keeping only git:  8720085


True

In [ ]:
origins_git = df['origin'].drop_duplicates().reset_index(drop=True)
print(f'amount of git origins with at least 1 root cause commits: {len(origins_git):,}')

amount of git origins with at least 1 root cause commits: 1,218,547


True

In [21]:
df.head()

,origin,snapshot_src,branch_name,missing_commit,snapshot_dst,first_difference,main_category,sub_categories
0,https://github.com/SonyaMoisset/SaaSApp_RUBY-ON-RAILS,swh:1:snp:a19832274d0cbfdba0c9986917bd09b018caa07c,refs/pull/78/head,swh:1:rev:f6afdb795b66d0a79d943e77770bb4f6e7e4379c,swh:1:snp:e5374f7e2f9dfd110bda69770057fecb0c2ee2ef,NaN,DIR,",FileModified"
2,https://github.com/SonyaMoisset/SaaSApp_RUBY-ON-RAILS,swh:1:snp:0dff15e79b3136ebf8974cb5a3250b1383ffb946,refs/pull/78/head,swh:1:rev:8b4699448f39b919fc28fcd7990edcae8890341f,swh:1:snp:ee97c5d760577fe489cb1cc134d733ed9dbf22ea,NaN,DIR,",FileModified"
4,https://github.com/mlesniak/mlesniak.github.io,swh:1:snp:aec585f39a63de81a8a5f3d221d7bd43ac5f892f,refs/heads/gh-pages,swh:1:rev:9965821916130e3d5bdde93c68a97bb34ec6654e,swh:1:snp:f553e234b4d1b92a50ef3f7fcbec00777d3495ae,NaN,DIR,",FileRemoved"
5,https://github.com/Pigmice2733/peregrine-frontend,swh:1:snp:d62caf130969027464f4e9e0918ffeae161dd014,refs/pull/1273/head,swh:1:rev:f398bfbd40943a38d7cf89787b3ed5857ab7a51e,swh:1:snp:b39ce059c3189a99cd3032859bda6b1e70165693,NaN,DIR,",FileModified"
6,https://github.com/erezrokah/aws-custom-resources,swh:1:snp:581e14d43dde27de11e53a496d02f8b28cde1e5b,refs/heads/renovate/axios-1.x,swh:1:rev:bd5eb3b881d8dee4deb3b516d899ba02d88901cf,swh:1:snp:24a0baa740a6cac9c9846469d8e97b255577d903,NaN,DIR,",FileModified"


In [13]:
star_dict = db_stars.set_index('origin')['stars'].to_dict()

In [24]:
df['stars'] = df['origin'].map(star_dict).fillna("Unknown")

In [25]:
df.head()

,origin,snapshot_src,branch_name,missing_commit,snapshot_dst,first_difference,main_category,sub_categories,stars
0,https://github.com/SonyaMoisset/SaaSApp_RUBY-ON-RAILS,swh:1:snp:a19832274d0cbfdba0c9986917bd09b018caa07c,refs/pull/78/head,swh:1:rev:f6afdb795b66d0a79d943e77770bb4f6e7e4379c,swh:1:snp:e5374f7e2f9dfd110bda69770057fecb0c2ee2ef,NaN,DIR,",FileModified",3.0
2,https://github.com/SonyaMoisset/SaaSApp_RUBY-ON-RAILS,swh:1:snp:0dff15e79b3136ebf8974cb5a3250b1383ffb946,refs/pull/78/head,swh:1:rev:8b4699448f39b919fc28fcd7990edcae8890341f,swh:1:snp:ee97c5d760577fe489cb1cc134d733ed9dbf22ea,NaN,DIR,",FileModified",3.0
4,https://github.com/mlesniak/mlesniak.github.io,swh:1:snp:aec585f39a63de81a8a5f3d221d7bd43ac5f892f,refs/heads/gh-pages,swh:1:rev:9965821916130e3d5bdde93c68a97bb34ec6654e,swh:1:snp:f553e234b4d1b92a50ef3f7fcbec00777d3495ae,NaN,DIR,",FileRemoved",0.0
5,https://github.com/Pigmice2733/peregrine-frontend,swh:1:snp:d62caf130969027464f4e9e0918ffeae161dd014,refs/pull/1273/head,swh:1:rev:f398bfbd40943a38d7cf89787b3ed5857ab7a51e,swh:1:snp:b39ce059c3189a99cd3032859bda6b1e70165693,NaN,DIR,",FileModified",25.0
6,https://github.com/erezrokah/aws-custom-resources,swh:1:snp:581e14d43dde27de11e53a496d02f8b28cde1e5b,refs/heads/renovate/axios-1.x,swh:1:rev:bd5eb3b881d8dee4deb3b516d899ba02d88901cf,swh:1:snp:24a0baa740a6cac9c9846469d8e97b255577d903,NaN,DIR,",FileModified",0.0


We stack together all branches e.g. `refs/pull/{digits}/head`

In [ ]:
unique_branch = df['branch_name'].drop_duplicates().reset_index()
print("Amount of branch name in the dataframe: ", len(unique_branch))

Amount of branch name in the dataframe:  1233793


True

In [4]:
def normalize_branch_name(branch_name):
    pull_pattern = r'refs/pull/\d+/head'
    if re.match(pull_pattern, branch_name):
        return re.sub(r'(?<=refs/pull/)\d+(?=/head)', '{x}', branch_name)
    
    renovate_pattern = r'refs/heads/renovate/.*'
    if re.match(renovate_pattern, branch_name):
        return re.sub(r'refs/heads/renovate/.*', 'refs/heads/renovate/{project}', branch_name)
    
    main_pattern = r'refs/heads/(main|master)'
    if re.match(main_pattern, branch_name):
        return 'refs/heads/{master,main}'
    
    dev_pattern = r'refs/heads/(dev|devel|develop|development)'
    if re.match(dev_pattern, branch_name):
        return 'refs/heads/{dev(el(op(ment)))}'
    
    return branch_name
df['normalized_branch'] = df['branch_name'].apply(normalize_branch_name)

In [ ]:
unique_branch = df['normalized_branch'].drop_duplicates().reset_index()
print("Amount of branch name in the dataframe after normalizing: ", len(unique_branch))

Amount of branch name in the dataframe after normalizing:  1115052


True

In [ ]:
df["sub_categories"] = df["sub_categories"].fillna(",Other")
df["sub_categories"] = df["sub_categories"].str.replace(",", " ").str.strip()

## Saving Results

In [5]:
df.head()

,origin,snapshot_src,branch_name,missing_commit,snapshot_dst,first_difference,main_category,sub_categories,stars,normalized_branch
0,https://github.com/SonyaMoisset/SaaSApp_RUBY-ON-RAILS,swh:1:snp:a19832274d0cbfdba0c9986917bd09b018caa07c,refs/pull/78/head,swh:1:rev:f6afdb795b66d0a79d943e77770bb4f6e7e4379c,swh:1:snp:e5374f7e2f9dfd110bda69770057fecb0c2ee2ef,NaN,DIR,FileModified,3.0,refs/pull/{x}/head
2,https://github.com/SonyaMoisset/SaaSApp_RUBY-ON-RAILS,swh:1:snp:0dff15e79b3136ebf8974cb5a3250b1383ffb946,refs/pull/78/head,swh:1:rev:8b4699448f39b919fc28fcd7990edcae8890341f,swh:1:snp:ee97c5d760577fe489cb1cc134d733ed9dbf22ea,NaN,DIR,FileModified,3.0,refs/pull/{x}/head
4,https://github.com/mlesniak/mlesniak.github.io,swh:1:snp:aec585f39a63de81a8a5f3d221d7bd43ac5f892f,refs/heads/gh-pages,swh:1:rev:9965821916130e3d5bdde93c68a97bb34ec6654e,swh:1:snp:f553e234b4d1b92a50ef3f7fcbec00777d3495ae,NaN,DIR,FileRemoved,0.0,refs/heads/gh-pages
5,https://github.com/Pigmice2733/peregrine-frontend,swh:1:snp:d62caf130969027464f4e9e0918ffeae161dd014,refs/pull/1273/head,swh:1:rev:f398bfbd40943a38d7cf89787b3ed5857ab7a51e,swh:1:snp:b39ce059c3189a99cd3032859bda6b1e70165693,NaN,DIR,FileModified,25.0,refs/pull/{x}/head
6,https://github.com/erezrokah/aws-custom-resources,swh:1:snp:581e14d43dde27de11e53a496d02f8b28cde1e5b,refs/heads/renovate/axios-1.x,swh:1:rev:bd5eb3b881d8dee4deb3b516d899ba02d88901cf,swh:1:snp:24a0baa740a6cac9c9846469d8e97b255577d903,NaN,DIR,FileModified,0.0,refs/heads/renovate/{project}


In [ ]:
with open('../../data/res.pkl', 'wb') as f:
    pickle.dump(df, f)